# atomicwrites Package Testing

This notebook tests the atomicwrites package - Atomic file write operations.

## Example 1: Basic atomic write operations


In [ ]:
import atomicwrites
import tempfile
import os

# Example 1: Basic atomic write operations
print("Example 1: Basic atomic write operations")

try:
    # Create a temporary directory for testing
    with tempfile.TemporaryDirectory() as temp_dir:
        test_file = os.path.join(temp_dir, "test_atomic.txt")
        
        # Basic atomic write
        with atomicwrites.atomic_write(test_file, overwrite=True) as f:
            f.write("Hello, atomic world!\n")
            f.write("This write is atomic.\n")
        
        # Verify the file was written
        with open(test_file, 'r') as f:
            content = f.read()
            print("File content after atomic write:")
            print(content)
        assert content == "Hello, atomic world!\nThis write is atomic.\n", repr(content)
        
        # Atomic write with different content
        with atomicwrites.atomic_write(test_file, overwrite=True) as f:
            f.write("Updated content\n")
            f.write("Still atomic!\n")
        
        # Verify the update
        with open(test_file, 'r') as f:
            updated_content = f.read()
            print("Updated file content:")
            print(updated_content)
        assert updated_content == "Updated content\nStill atomic!\n", repr(updated_content)
        assert os.listdir(temp_dir) == ["test_atomic.txt"], os.listdir(temp_dir)  # no temporary files left
        
        print("✓ Atomic write operations completed successfully")
    
    print("Basic atomic write operations completed successfully")
    
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"atomicwrites example (cell 2) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 1 finished")

## Example 2: Failure handling and rollback


In [ ]:
import atomicwrites
import tempfile
import os

# Example 2: Failure handling and rollback
print("Example 2: Failure handling and rollback")

try:
    with tempfile.TemporaryDirectory() as temp_dir:
        test_file = os.path.join(temp_dir, "test_rollback.txt")
        
        # Create initial content
        with open(test_file, 'w') as f:
            f.write("Original content\n")
        
        print("Original content:", open(test_file).read().strip())
        
        # Simulate failure during atomic write
        try:
            with atomicwrites.atomic_write(test_file, overwrite=True) as f:
                f.write("Partial write...\n")
                # Simulate an error
                raise ValueError("Simulated error during write")
        except ValueError as e:
            print(f"Error occurred: {e}")
        
        # Verify original content is preserved
        with open(test_file, 'r') as f:
            preserved_content = f.read()
            print("Content after failed write:", preserved_content.strip())
        assert preserved_content == "Original content\n", repr(preserved_content)
        assert os.listdir(temp_dir) == ["test_rollback.txt"], os.listdir(temp_dir)  # the partial temporary file was removed
            
        print("✓ Original content preserved after failure")
        
    print("Failure handling and rollback completed successfully")
    
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"atomicwrites example (cell 4) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 2 finished")

## Example 3: Binary file operations


In [ ]:
import atomicwrites
import tempfile
import os

# Example 3: Binary file operations
print("Example 3: Binary file operations")

try:
    with tempfile.TemporaryDirectory() as temp_dir:
        binary_file = os.path.join(temp_dir, "test_binary.dat")
        
        # Create binary data
        binary_data = bytes([i for i in range(256)])
        
        # Atomic write of binary data
        with atomicwrites.atomic_write(binary_file, mode='wb', overwrite=True) as f:
            f.write(binary_data)
        
        # Verify binary data
        with open(binary_file, 'rb') as f:
            read_data = f.read()
            
        print(f"Original data length: {len(binary_data)}")
        print(f"Read data length: {len(read_data)}")
        print(f"Data matches: {binary_data == read_data}")
        assert read_data == binary_data
        
        print("✓ Binary file operations completed successfully")
        
    print("Binary file operations completed successfully")
    
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"atomicwrites example (cell 6) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 3 finished")

## Example 4: Replacing an existing file, and its permissions


In [ ]:
import atomicwrites
import tempfile
import os
import stat

# Example 4: Replacing an existing file, and its permissions
print("Example 4: Replacing an existing file, and its permissions")

try:
    with tempfile.TemporaryDirectory() as temp_dir:
        perm_file = os.path.join(temp_dir, "test_permissions.txt")
        
        # Create file with content, readable by its group
        with open(perm_file, 'w') as f:
            f.write("Original content with permissions\n")
        os.chmod(perm_file, 0o640)
        before = os.stat(perm_file)
        
        # Atomic write with new content
        with atomicwrites.atomic_write(perm_file, overwrite=True) as f:
            f.write("Updated content\n")
        
        # Verify content was updated
        with open(perm_file, 'r') as f:
            content = f.read()
            print("Updated content:", content.strip())
        assert content == "Updated content\n", repr(content)
        
        after = os.stat(perm_file)
        print(f"Mode before: {oct(stat.S_IMODE(before.st_mode))}, after: {oct(stat.S_IMODE(after.st_mode))}")
        if os.name == "posix":
            # The new content is written to a temporary file that is renamed over the old one:
            # the path now names a different file, created owner-only like any tempfile.mkstemp file
            assert after.st_ino != before.st_ino
            assert stat.S_IMODE(after.st_mode) == 0o600, oct(after.st_mode)
            # Permissions are not carried over, so restore them after the write if they matter
            os.chmod(perm_file, stat.S_IMODE(before.st_mode))
            assert stat.S_IMODE(os.stat(perm_file).st_mode) == 0o640
            print("✓ Old permissions restored with os.chmod after the atomic replace")
        
    print("Replacing an existing file completed successfully")
    
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"atomicwrites example (cell 8) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 4 finished")

## Example 5: Different write modes and encoding


In [ ]:
import atomicwrites
import tempfile
import os

# Example 5: Different write modes and encoding
print("Example 5: Different write modes and encoding")

try:
    with tempfile.TemporaryDirectory() as temp_dir:
        # Test different encodings
        encodings = ['utf-8', 'latin1', 'ascii']
        
        for encoding in encodings:
            test_file = os.path.join(temp_dir, f"test_{encoding}.txt")
            
            # Content with encoding-specific characters
            if encoding == 'utf-8':
                content = "Hello world! 🌍"
            elif encoding == 'latin1':
                content = "Hello café"
            else:  # ascii
                content = "Hello world ASCII"
            
            # Atomic write with specific encoding
            with atomicwrites.atomic_write(test_file, mode='w', 
                                         encoding=encoding, overwrite=True) as f:
                f.write(content)
            
            # Verify with same encoding
            with open(test_file, 'r', encoding=encoding) as f:
                read_content = f.read()
                
            print(f"{encoding}: {content == read_content} - {read_content}")
            assert read_content == content, (encoding, read_content)
        
        print("✓ Different encodings handled successfully")
        
        # overwrite=False (the default) refuses to replace an existing file and leaves it untouched
        existing = os.path.join(temp_dir, "test_utf-8.txt")
        try:
            with atomicwrites.atomic_write(existing, encoding='utf-8') as f:
                f.write("must not replace")
        except FileExistsError:
            print("✓ overwrite=False refused to replace an existing file")
        else:
            raise AssertionError("atomic_write replaced an existing file without overwrite=True")
        with open(existing, encoding='utf-8') as f:
            assert f.read() == "Hello world! 🌍"
        assert sorted(os.listdir(temp_dir)) == [f"test_{e}.txt" for e in sorted(encodings)], os.listdir(temp_dir)
        
    print("Different write modes and encoding completed successfully")
    
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"atomicwrites example (cell 10) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 5 finished")